# Rain in Australia

## 1. Skup podataka

Ovaj rad se bavi problemom binarne klasifikacije – predviđanjem da li će sledećeg dana padati kiša na određenoj lokaciji u Australiji, na osnovu skupa podataka Rain in Australia koji sadrži preko 145.000 dnevnih meteoroloških merenja sa 49 stanica u periodu 2007–2017. godine. Istraživanje obuhvata ceo tok obrade podataka: detekciju strukturalnih i fizičkih grešaka u merenjima, detaljnu eksplorativnu i statističku analizu obeležja (univarijantnu i multivarijantnu, uključujući Pirsonovu i Spirmanovu korelaciju i VIF faktor), višeslojnu detekciju anomalija (Z-skor zasnovan na MAD-u, IQR, Mahalanobisova udaljenost, Isolation Forest, LOF i KNN) i inženjering obeležja (ciklično kodiranje vremenskih promenljivih, geografske i klimatske karakteristike stanica, lag i pokretni proseci). Poseban deo rada posvećen je popunjavanju nedostajućih vrednosti, koje se u pojedinim kolonama (Sunshine, Evaporation, Cloud9am/3pm) javljaju kod čak 40–48% opažanja, pri čemu je primenjena kombinacija prostorne imputacije po grupama susednih stanica i Random Forest modela, čiji je kvalitet potvrđen R² metrikom (do 0.98) i vizuelnim poređenjem raspodela. Značaj atributa je procenjen kombinovanjem Random Forest feature importance i mutual information mere.
Nakon uklanjanja nedostajućih ciljnih vrednosti, konačan skup za modelovanje je obuhvatao 133.283 opservacije, hronološki podeljene (granica 10.11.2015.) na 107.152 opservacije za trening i 26.131 za test. Upoređeni su model odlučivanja (Decision Tree), Random Forest i XGBoost, uz balansiranje klasa i optimizaciju praga odlučivanja radi maksimizacije F1 ocene za manjinsku klasu (dani sa kišom), s obzirom na izraženu neuravnoteženost skupa (~77% dana bez kiše). Najbolje rezultate ostvario je Random Forest sa balansiranim klasama i optimizovanim pragom od 0.42, sa tačnošću od 82.3%, ROC-AUC vrednošću od 0.875 i F1 ocenom od 0.65 za klasu kiše (preciznost 0.59, odziv 0.73), dok je XGBoost sa uravnoteženim težinama ostvario najviši ROC-AUC (0.879) uz sličan F1 (0.65). Za poređenje, XGBoost sa podrazumevanim pragom od 0.50 (bez balansiranja) postigao je najvišu ukupnu tačnost (85.0%), ali znatno niži odziv za kišne dane (54%), što potvrđuje očekivani kompromis između preciznosti i odziva kod neuravnoteženih klasa. Praćenje eksperimenata i produkciono servisiranje najboljeg modela realizovano je pomoću alata MLflow.

## 2. Priprema razvojnog okruženja

U folderu req nalazi se spisak svih biblioteka potrebnih za izvršavanje celokupne dalje analize, zajedno sa potrebnim verzijama. 

Pokrenućemo pip install komandu koja će obezbediti da na aktuelnoj sesiji imamo instalirane sve potrebne pakete.


In [ ]:
import os
currentDir = os.getcwd()
!pip install -r "{currentDir}/req/requirements.txt"

Zbog preglednosti ostatka projekta, sada ćemo takođe importovati sve biblioteke, na taj način u ostatku koda ćemo imati potpuno pripremljeno razvojno okruženje i možemo se fokusirati isključivo na logiku projekta.

In [ ]:
from common import *

## 3. Konfiguracija pomoćnih servisa

### 3.1 Vođenje logova

Analiza u nastavku će iziskivati veliki broj modifikacija i manipulacija skupom podataka, iz tog razloga kreiraćemo funkciju koja će pri pozivu automatski beležiti sve promene koje su se desile na skupu podataka od poslednjeg poziva metode, ove promene biće sačuvane u log folderu.

Takođe izrvšavanje celokupnog programa svaki put bi uzimalo veliku količinu vremena, zbog toga ćemo nakon svakog koraka i loga koji izvršimo automatski čuvati backup dataset-a, na početku svakog sledećeg koraka možemo samo učitati csv fajl generisan u logu prethodnog koraka.

### 3.2 Mlflow

Za praćenje različitih modela, njihovih ocena kao i hiperparametara koristićemo mlflow, u pitanju je open-source python biblioteka koja kreira lokalni server na kojem se čuvaju sve navedene informacije o svakom eksperimentu (svakom istreniranom modelu).

Još jedna velika prednost mlflow-a je mogućnost brzog i jednostavnog stavljanja svakog od istreniranih modela u produkciju, upotrebom baš ovog alata ćemo naš najbolji model na kraju podići u zaseban lokalni server sa kojim ćemo dalje komunicirati pomoću REST protokola.

Za pokretanje mlflow koristićemo docker.

In [ ]:
import os
trenutni_folder = os.getcwd()

model_id = "m-d4ab200d16a844409336449facd20ae5"

# 1. Ocistiti stare kontejnere ako postoje
!docker rm -f mlflow_tracking_server

# 2. Pokrenuti tracking server (port 5000)
!docker run -d -p 5000:5000 --name mlflow_tracking_server -v "{trenutni_folder}:/workspace" -w /workspace -e MLFLOW_ALLOW_FILE_STORE=true python:3.12-slim bash -c "pip install mlflow && mlflow server -h 0.0.0.0 --allowed-hosts 'localhost:*,127.0.0.1:*,host.docker.internal:*'"